In [ ]:
%%bash
cd ..
# rm data/*
# rm -r logs
# rm -r src/__pycache__
tree

In [1]:
# Python modules
import os
import pickle
import warnings

from datetime import datetime, timezone
from importlib import reload
from pathlib import PosixPath

import hopsworks
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px

from catboost import CatBoostRegressor
from dotenv import load_dotenv
from hopsworks.project import Project
from hsfs.feature_group import FeatureGroup
from hsfs.feature_store import FeatureStore
from hsml.model_registry import ModelRegistry
from hsml.model_schema import ModelSchema, Schema
from hsml.sklearn.model import Model
from lightgbm import LGBMRegressor
from plotly.graph_objects import Figure
from xgboost import XGBRegressor


# taxi-demand-forecasting modules
import src.config
import src.feature_store_api
import src.inference
import src.ingest
import src.pipelines.feature_pipeline
import src.pipelines.training_pipeline
import src.train
import src.transform
import src.utils

reload(src.config)
reload(src.feature_store_api)
reload(src.inference)
reload(src.ingest)
reload(src.pipelines.feature_pipeline)
reload(src.pipelines.training_pipeline)
reload(src.train)
reload(src.transform)
reload(src.utils)

from src.config import Config
from src.feature_store_api import get_feature_group, HOPSWORKS_CONFIG
from src.inference import generate_forecast
from src.ingest import download_file, fetch_and_preprocess
from src.pipelines.feature_pipeline import upload_data
from src.pipelines.training_pipeline import evaluate_model, upload_model
from src.train import compute_metrics, NaiveForecast, split_data, train_model
from src.transform import fetch_and_transform, tabularize_data
from src.utils import load_taxi_zones, plot_record

warnings.filterwarnings("ignore")
load_dotenv(Config.HOME_DIR / ".env")

True

In [2]:
# set the Pandas display options
pd.set_option("display.max_rows", 100) # max number of rows to display
pd.set_option("display.max_columns", None) # max number of columns to display

# set the Polars display options
pl.Config(tbl_rows=10) # max number of rows to display
pl.Config(tbl_cols=None) # max number of columns to display
pl.Config(fmt_str_lengths=25) # max number of characters to display for pl.Utf8 (str) dtype columns
pl.Config(fmt_table_cell_list_len=20) # max number of items to display for pl.List dtype columns

#### **`Data ingestion`**

In [ ]:
# download, validate, and pre-process raw data from 2022-01
df: pd.DataFrame = download_file(2022, 1)
df

In [ ]:
# confirm that the 'df' pd.DataFrame is free of null values and duplicates
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0

In [ ]:
# a list of select location IDs
location_ids: list[int] = [43, 90, 107]

# plot the hourly taxi rides for the selected location IDs
fig: Figure = px.line(
    df.query(f"location_id.isin({location_ids})"),
    x="pickup_datetime",
    y="rides",
    color="location_id",
    labels={
        "pickup_datetime": "Datetime", 
        "rides": "Number of taxi rides", 
        "location_id": "Location ID"
    },
    title="NYC Hourly Taxi Rides",
    template="plotly_dark"
)
fig.show()

#### **`Data transformation`**

In [ ]:
# download, validate, pre-process, and transform raw data from 2022-01 into features and labels
# NOTE: for the 'day_of_week' feature, 0 == Monday, 1 == Tuesday, ..., 6 == Sunday
df: pd.DataFrame = download_file(2022, 1).pipe(tabularize_data)
df

In [ ]:
# plot a single record's (row's) lag features (blue) and target (green)
plot_record(df, np.random.choice(location_ids))

#### **`Model training and evaluation`**

In [ ]:
# download, validate, pre-process, and transform raw data from 2023-05 into an ML-ready dataset
df: pd.DataFrame = download_file(2023, 5).pipe(tabularize_data)

In [ ]:
# extract the 'best' model
model: CatBoostRegressor | LGBMRegressor | XGBRegressor = train_model(df)

# split the 'df' pd.DataFrame into train and test sets
Xtrain, ytrain, Xtest, ytest = split_data(df)

# fit the model to the train set and evaluate on the test set
if isinstance(model, CatBoostRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], early_stopping_rounds=50, verbose=False)
elif isinstance(model, LGBMRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)])
elif isinstance(model, XGBRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], verbose=False)

# output the model and its corresponding test set metrics (RMSE and R²)
display(model, compute_metrics(ytest, model.predict(Xtest)))

#### **`Feature pipeline (Hopsworks)`**

In [ ]:
# specify the starting and ending timestamps for which to ingest data
current_ts: pd.Timestamp = pd.Timestamp(datetime.now(timezone.utc)).floor("H")
end: pd.Timestamp = current_ts - pd.Timedelta(days=366)
start: pd.Timestamp = end - pd.Timedelta(days=28)

# a list of pre-processed pd.DataFrames, one per (year, month) pair
dfs: list[pd.DataFrame] = [
    download_file(year, month) 
    for year, month in zip([start.year, end.year], [start.month, end.month])
]

# (1) convert each pd.DataFrame in the 'dfs' list to a pl.DataFrame, and ...
# vertically concatenate them into a single pl.DataFrame
# (2) modify the 'pickup_datetime' column's timestamps by setting their timezone to UTC.
# Then, filter the pl.DataFrame by only keeping records (rows) whose modified timestamp is ...
# greater than or equal to the 'start' timestamp and less than or equal to the 'end' timestamp
# (3) modify the 'pickup_datetime' column's timestamps by ...
# (a) setting their timezone to UTC, ...
# (b) offsetting them forward in time by one year, and ...
# (c) converting them to integers that represent the time passed, in ms, since the Unix EPOCH
# (4) rename the 'pickup_datetime' column to 'unix_epoch_ms'
# (5) convert the pl.DataFrame to a pd.DataFrame
# NOTE: the NYC taxi data lags behind the current date by three months, so the timestamps are ...
# offset forward in time by one year to simulate a real time data ingestion scenario
# NOTE: the 'df' pd.DataFrame contains the data that will be uploaded to Hopsworks
df: pd.DataFrame = (
    pl.concat([pl.from_pandas(df) for df in dfs], how="vertical") # (1)
    .filter( # (2)
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").ge(start),
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").le(end)
    )
    .with_columns( # (3)
        pl.col("pickup_datetime")
        .dt.replace_time_zone("UTC")
        .dt.offset_by("1y")
        .dt.epoch(time_unit="ms")
    )
    .rename({"pickup_datetime": "unix_epoch_ms"}) # (4)
    .to_pandas() # (5)
)
df

In [ ]:
# NOTE: this cell transforms the 'df' pd.DataFrame to an ML-ready dataset that contains ...
# datetime features, window features (average lags), lag features, and the corresponding target
# (1) convert the 'df' pd.DataFrame to a pl.DataFrame
# (2) convert the 'unix_epoch_ms' column's entries, which are integers, to timestamps
# (3) rename the 'unix_epoch_ms' column to 'pickup_datetime'
# (4) sort the pl.DataFrame by the 'location_id' and 'pickup_datetime' columns
# (5) convert the pl.DataFrame back to a pd.DataFrame
# (6) tabularize the pd.DataFrame, that is, convert it to an ML-ready dataset containing ...
# datetime features, window features (average lags), lag features, and the corresponding target
(
    pl.from_pandas(df)
    .with_columns(pl.from_epoch(pl.col("unix_epoch_ms"), time_unit="ms"))
    .rename({"unix_epoch_ms": "pickup_datetime"})
    .sort(["location_id", "pickup_datetime"])
    .to_pandas()
    .pipe(tabularize_data)
)

In [ ]:
# connect to the Hopsworks 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project="taxi_demand_forecasting",
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# connect to the project's feature store
feature_store: FeatureStore = project.get_feature_store()

# create the feature store's feature group
# NOTE: the feature group allows the 'df' pd.DataFrame's features to be saved to the feature store
# NOTE: the 'primary_key' parameter below, which can be set equal to a single column or ...
# a list of columns from the 'df' pd.DataFrame, is used as a unique identifier for each record (row)
feature_group: FeatureGroup = feature_store.get_or_create_feature_group(
    name="univariate_time_series",
    version=1,
    description="NYC taxi rides recorded at an hourly frequency",
    primary_key=["location_id", "unix_epoch_ms"],
    event_time="unix_epoch_ms",
    online_enabled=True
)

# upload the 'df' pd.DataFrame to the 'taxi_demand_forecasting' project as a new feature group
# NOTE: the feature group's name is, 'univariate_time_series', and it can be found on the ...
# Hopsworks UI, under the 'taxi_demand_forecasting' project's 'Feature Group' tab 
feature_group.insert(df, write_options={"wait_for_job": False})

#### **`Training pipeline (Hopsworks)`**

In [ ]:
# fetch the pre-processed data from Hopsworks
processed_data: pd.DataFrame = get_feature_group().read()

# convert the pre-processed data to tabular, ML-ready features and labels
if not processed_data.empty:
    tabular_data: pd.DataFrame = (
        pl.from_pandas(processed_data)
        .with_columns(pl.from_epoch(pl.col("unix_epoch_ms"), time_unit="ms"))
        .rename({"unix_epoch_ms": "pickup_datetime"})
        .sort(["location_id", "pickup_datetime"])
        .to_pandas()
        .pipe(tabularize_data)
    )
tabular_data

In [ ]:
# split the 'tabular_data' pd.DataFrame into train and test sets
Xtrain, ytrain, Xtest, ytest = tabular_data.pipe(split_data)

# get the test set's 'baseline' metrics
baseline_metrics: dict[str, float] = compute_metrics(ytest, NaiveForecast().predict(Xtest))

# fit the model to the train set and evaluate on the test set
if isinstance(model, CatBoostRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], early_stopping_rounds=50, verbose=False)
elif isinstance(model, LGBMRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)])
elif isinstance(model, XGBRegressor):
    model.fit(Xtrain, ytrain, eval_set=[(Xtest, ytest)], verbose=False)
    
# get the model's test set metrics
metrics: dict[str, float] = compute_metrics(ytest, model.predict(Xtest))

# display the model, its test set metrics, and the test set's 'baseline' metrics
display(model, f"Metrics: {metrics}", f"Baseline metrics: {baseline_metrics}")

In [ ]:
# if the model's test set predictions are better than the naive forecast, then ...
# (1) save the model locally as an artifact, that is, ~/artifacts/model.pkl, and ...
# (2) upload the saved artifact to the Hopsworks model registry
if metrics.get("rmse") < baseline_metrics.get("rmse"):
    output_dir: PosixPath = Config.ARTIFACTS_DIR
    output_dir.mkdir(parents=True, exist_ok=True)
    pickle.dump(model, open(output_dir / "model.pkl", "wb"))

# confirm that the saved model works
# NOTE: the metrics output from this cell should match the 'Metrics' dict from the above code cell
saved_model: LGBMRegressor = pickle.load(open(Config.ARTIFACTS_DIR / "model.pkl", "rb"))
compute_metrics(ytest, saved_model.predict(Xtest))

In [ ]:
%%bash
cd ..
tree artifacts

In [ ]:
(
    project
    .get_model_registry()
    .get_model(name=HOPSWORKS_CONFIG.get("model_registry").get("model_name"), version=1)
)

In [ ]:
# connect to the Hopsworks 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project="taxi_demand_forecasting",
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# create an object that points to the project's model registry
model_registry: ModelRegistry = project.get_model_registry()

# upload ~/artifacts/model.pkl to the project's model registry
# NOTE: the model and its metadata can be viewed on the Hopsworks UI, under the 'taxi_demand_forecasting' 
# project's 'Model Registry' tab 
(
    model_registry.sklearn
    .create_model(
        name="one_step_forecaster",
        metrics={"rmse": metrics.get("rmse")},
        metrics=metrics,
        description=model.__str__(),
        input_example=Xtrain.sample(),
        model_schema=ModelSchema(input_schema=Schema(Xtrain), output_schema=Schema(ytrain))
    )
    .save(os.path.join(Config.ARTIFACTS_DIR, "model.pkl"))
)

In [ ]:
# add the 'location_id', 'pickup_datetime', and 'target' columns back to the train set features
tabular_data_train: pd.DataFrame = (
    Xtrain.assign(
        location_id=tabular_data.loc[Xtrain.index, "location_id"],
        pickup_datetime=tabular_data.loc[Xtrain.index, "pickup_datetime"], 
        target=ytrain
    )
    [tabular_data.columns]
)

# add the 'location_id', 'pickup_datetime', and 'target' columns back to the test set features
tabular_data_test: pd.DataFrame = (
    Xtest.assign(
        location_id=tabular_data.loc[Xtest.index, "location_id"],
        pickup_datetime=tabular_data.loc[Xtest.index, "pickup_datetime"], 
        target=ytest
    )
    [tabular_data.columns]
)

# vertically concatenate the 'tabular_data_train' and 'tabular_data_test' pd.DataFrames
tabular_data_confirm: pd.DataFrame = (
    pd.concat((tabular_data_train, tabular_data_test), axis=0).sort_index()
)

# confirm that the 'tabular_data' and 'tabular_data_confirm' pd.DataFrames are identical
assert np.mean(tabular_data.values == tabular_data_confirm.values) == 1

#### **`Inference pipeline and Streamlit web app`**

In [ ]:
# (1) fetch the latest hourly NYC taxi demand data from the Hopsworks project's Feature Group
# (2) transform the hourly NYC taxi demand data into ML-ready features and labels
# (3) load the model from the Hopsworks project's Model Registry
# (4) use the model to generate the one-step forecast, i.e., the taxi demand for the next hour

In [ ]:
# specify the starting and ending timestamps for which to ingest data
current_ts: pd.Timestamp = pd.Timestamp(datetime.now(timezone.utc)).floor("H")
end: pd.Timestamp = current_ts - pd.Timedelta(days=366)
start: pd.Timestamp = end - pd.Timedelta(days=28)

# a list of pre-processed pd.DataFrames, one per (year, month) pair
dfs: list[pd.DataFrame] = [
    download_file(year, month) 
    for year, month in zip([start.year, end.year], [start.month, end.month])
]

# (1) convert each pd.DataFrame in the 'dfs' list to a pl.DataFrame, and ...
# vertically concatenate them into a single pl.DataFrame
# (2) modify the 'pickup_datetime' column's timestamps by setting their timezone to UTC.
# Then, filter the pl.DataFrame by only keeping records (rows) whose modified timestamp is ...
# greater than or equal to the 'start' timestamp and less than or equal to the 'end' timestamp
# (3) modify the 'pickup_datetime' column's timestamps by ...
# (a) setting their timezone to UTC, ...
# (b) offsetting them forward in time by one year, and ...
# (c) converting them to integers that represent the time passed, in ms, since the Unix EPOCH
# (4) rename the 'pickup_datetime' column to 'unix_epoch_ms'
# (5) convert the pl.DataFrame to a pd.DataFrame
# NOTE: the NYC taxi data lags behind the current date by three months, so the timestamps are ...
# offset forward in time by one year to simulate a real time data ingestion scenario
# NOTE: the 'df' pd.DataFrame contains the data that will be uploaded to Hopsworks
df: pd.DataFrame = (
    pl.concat([pl.from_pandas(df) for df in dfs], how="vertical") # (1)
    .filter( # (2)
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").ge(start),
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").le(end)
    )
    .with_columns( # (3)
        pl.col("pickup_datetime")
        .dt.replace_time_zone("UTC")
        .dt.offset_by("1y")
        .dt.epoch(time_unit="ms")
    )
    .rename({"pickup_datetime": "unix_epoch_ms"}) # (4)
    .to_pandas() # (5)
)
df

In [ ]:
# connect to the Hopsworks 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project="taxi_demand_forecasting",
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# connect to the project's feature store
feature_store: FeatureStore = project.get_feature_store()

# create the feature store's feature group
# NOTE: the feature group allows the 'df' pd.DataFrame's features to be saved to the feature store
# NOTE: the 'primary_key' parameter below, which can be set equal to a single column or ...
# a list of columns from the 'df' pd.DataFrame, is used as a unique identifier for each record (row)
feature_group: FeatureGroup = feature_store.get_or_create_feature_group(
    name="univariate_time_series",
    version=1,
    description="NYC taxi rides recorded at an hourly frequency",
    primary_key=["location_id", "unix_epoch_ms"],
    event_time="unix_epoch_ms",
    online_enabled=True
)

# # upload the 'df' pd.DataFrame to the 'taxi_demand_forecasting' project as a new feature group
# # NOTE: the feature group's name is, 'univariate_time_series', and it can be found on the ...
# # Hopsworks UI, under the 'taxi_demand_forecasting' project's 'Feature Group' tab 
feature_group.insert(df, write_options={"wait_for_job": False})

In [ ]:
# output an ML-ready dataset of features and labels from the latest NYC taxi demand data
fetch_and_transform()

In [ ]:
# output the 10 busiest location IDs, based on their forecasted taxi demand
df_forecast: pd.DataFrame = (
    fetch_and_transform()
    .pipe(generate_forecast)
    .sort_values("forecast", ascending=False)
    .reset_index(drop=True)
    .head(10)
)
pl.from_pandas(df_forecast)

In [ ]:
# display a line plot of one of the 10 busiest location IDs
fig: Figure = plot_record(df_forecast, np.random.choice(df_forecast["location_id"]))
fig.show()

In [ ]:
# output the gpd.GeoDataFrame containing the NYC taxi zones
load_taxi_zones()

#### **`Hopsworks QC`**

In [3]:
# fetch the latest raw 'taxi rides' data, validate, pre-process, and upload it to Hopsworks
upload_data()

2024-07-26 17:37:24,851 INFO: Downloading, validating, and pre-processing https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-06.parquet


100%|██████████| 258/258 [00:04<00:00, 57.68it/s]


2024-07-26 17:37:39,602 INFO: Downloading, validating, and pre-processing https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-07.parquet


100%|██████████| 255/255 [00:03<00:00, 65.02it/s]

2024-07-26 17:37:49,125 INFO: Uploading NYC taxi demand data to Hopsworks, Project Name: taxi_demand_forecasting, Feature Group: univariate_time_series
Connected. Call `.close()` to terminate connection gracefully.



Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/903316
Connected. Call `.close()` to terminate connection gracefully.


Uploading Dataframe: 0.00% |          | Rows 0/162437 | Elapsed Time: 00:00 | Remaining Time: ?

Launching job: univariate_time_series_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai/p/903316/jobs/named/univariate_time_series_1_offline_fg_materialization/executions


In [4]:
# train, evaluate, select the 'best' ML model, and upload it to Hopsworks
upload_model()

Connection closed.
Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/903316
Connected. Call `.close()` to terminate connection gracefully.
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.70s) 
2024-07-26 17:38:24,366 INFO: Transforming the hourly taxi demand data into features and labels.


  0%|          | 0/3 [00:00<?, ?it/s]

2024-07-26 17:38:33,208 INFO: Training initiated for the CatBoostRegressor.


 33%|███▎      | 1/3 [00:05<00:11,  5.66s/it]

2024-07-26 17:38:38,873 INFO: Training initiated for the LGBMRegressor.


...

  0%|          | 0/6 [00:00<?, ?it/s]

Uploading: 0.000%|          | 0/184764 elapsed<00:00 remaining<?

Uploading: 0.000%|          | 0/203 elapsed<00:00 remaining<?

Uploading: 0.000%|          | 0/2408 elapsed<00:00 remaining<?

Model created, explore it at https://c.app.hopsworks.ai:443/p/903316/models/forecasting_model/1


In [5]:
# QC the current forecasting model
evaluate_model()

Connection closed.
Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/903316
Connected. Call `.close()` to terminate connection gracefully.
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.67s) 
2024-07-26 17:39:16,810 INFO: Transforming the hourly taxi demand data into features and labels.


100%|██████████| 254/254 [00:08<00:00, 28.70it/s]

Connection closed.
Connected. Call `.close()` to terminate connection gracefully.



Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/903316
Connected. Call `.close()` to terminate connection gracefully.


100%|██████████| 253/253 [00:00<00:00, 432.65it/s]


2024-07-26 17:39:29,354 INFO: The current forecasting model is fine.
